### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

• Tracking agent behavior with logging, analytics, and debugging.

• Transforming prompts, tool selection, and output formatting.

• Adding retries, fallbacks, and early termination logic.

• Applying rate limits, guardrails, and PII detection.

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

• Long-running conversations that exceed context windows.

• Multi-turn dialogues with extensive history.

• Applications where preserving full conversation context matters.

In [ ]:
# trigger was based on message size

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from  langchain_core.messages import HumanMessage,SystemMessage

### Message Based Summarization
agent = create_agent(
    model  = "google_genai:gemini-3.5-flash-lite",
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model  = "google_genai:gemini-3.5-flash-lite",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [17]:
### Run with thread_id

config={"configurable":{"thread_id":"test-1"}}

In [18]:
# Alternative Test data
questions= [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='69596a21-2dae-4ff2-9e16-d2191b740609'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAFpFH0TOy+1kmUC6PaJBMjpIlLEqOCP4eMvmbcBn1v+f39HaxW3HlA5tsQktC94Z9OPCtnVEqdMwknrRtZbDIO5uRKvXxSB3ESWCwXPS+wCtmCUbfX0VNpJrvcn'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0c92c-63f3-7a32-849b-8ba6adfe663d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='69596a21-2dae-4ff2-9e16-d2191b740609'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAFpFH0TOy+1kmU

In [19]:
# trigger based on token size
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from  langchain_core.messages import HumanMessage

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night,spa,pool,gym
    2.City Inn - 4 star, $180/night,business center
    3.Budget Stay - 3 star, $75/night,free wifi """

agent = create_agent(
    model  = "google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model  = "google_genai:gemini-3.5-flash-lite",
            trigger=("tokens",550),
            keep=("messages",200)
        ),
    ]
)

config={"configurable":{"thread_id":"test-1"}}

#Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 #4 chars = 1 token

In [20]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~173 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='15cff023-43b8-4d3f-84f2-a813bb86dd89'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_367052': 'El4KXAFpFH0TJFRPNuR6szfXLpLDeWhlOfFpV8k0Epf7uteZXmqdl66ybr4Jtl3f3qIpgZ1wX+GEDmJU1lYzDSjvCjAgUhUDSUDJndp4MRyyx1+zBOYFiAmVZjGM00yr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0c93a-152f-77e2-957e-a8ba2e9bd77c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_367052', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/

In [4]:
### Based on Fractions

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~102 tokens (0.0797%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='bb457c2e-f4ce-4ed4-b9fb-b251edfc38ec'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_361607': 'El4KXAFpFH0TTJCFNkZ05CydZNmA27MxzC+m3ObLk5AaZRmTmWcd+MowW7ztfyIqnTMIif+AOQrvgWHMw3R83y3NlNW/oWwiRT1yNkdkTsOQa66JRR0RdQrpsU2BdBHt'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1cc-e21d-7f02-b9f0-d558a58ec129-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_361607', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, B

KeyboardInterrupt: 

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).

Compliance workflows where human oversight is mandatory.

Long-running conversations where human feedback guides the agent.

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"



In [28]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer= InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [19]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1dcdd50c-24df-4246-ae56-72e6e7f9ecd2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_342933': 'El4KXAFpFH0T69W33puOqm+2uqRM2pvkX+wlFNkulwA70aAoQKG6lOJcH+1b+2kGwRJb/dg9Cf4nfcgY7xaKzhfCg6yJHIDShrTIhoDIB2RG1C4QMbFFg1KhrljChTDg'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1d2-4ce3-7a21-8417-5c2b70f484c8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'call_342933', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

In [21]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "Email successfully sent to john@test.com with subject 'Hello'.", 'extras': {'signature': 'El4KXAFpFH0TrSPjqqc0mchoveJRFJUoYOvT9HfZqU6Z3b8Ibhcw22NdEsn+o1LmtI9TkuNHMC53Rjz1Zs2vG/KR8ALHQsA0awecVvxYIvgYSUHQ3RfeT1ZlEHZ/kvcT'}}]


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1dcdd50c-24df-4246-ae56-72e6e7f9ecd2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_342933': 'El4KXAFpFH0T69W33puOqm+2uqRM2pvkX+wlFNkulwA70aAoQKG6lOJcH+1b+2kGwRJb/dg9Cf4nfcgY7xaKzhfCg6yJHIDShrTIhoDIB2RG1C4QMbFFg1KhrljChTDg'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1d2-4ce3-7a21-8417-5c2b70f484c8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'call_342933', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

## Reject

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [32]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [33]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email sending was rejected by the user. Let me know if you need assistance with anything else!', 'extras': {'signature': 'El4KXAFpFH0THsBVPyP1zFxCBYzhyEVfGgr564NSbsC54CbRF1Y69XOSI/05fV91gDCz5q0TBpyvWXW71zF9aGVxxreckuD6bNVO42arc8+iutbSIdaMYzKf11skCwEC'}}]


In [34]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='92a771f9-d5e1-4479-bf2a-7d45f23d5125'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_376942': 'El4KXAFpFH0TSr7oYlgfUMCv3g62Er5RB/1gLe36GunUXkFUJSz0BBfYc6fZfXFs/mbhbrSskSHMToccPLvZT20/1LuvFGiHHBfX+hyJSOv5hMPZbLc9Zh+o8UQK3qZ0'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1d7-4148-7ac1-a9a9-a6fd1a4912e4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_376942', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

## Editing

In [35]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [36]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [37]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='fa5a0ac0-9946-48d5-9a10-2b8b7e5c1bb0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "recipient": "wrong@email.com", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_331458': 'El4KXAFpFH0TJfddnbAKgeYkZV74gTAJiCtHm9MrLsWnUx6awV8eVyeUulNDiJY3LpspWX7WuuKSi0DU/6xvgZ5d7CrDzzqfFP0PRMq7ETDSpn9N3A7jlMAvDV0VofaU'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1d8-4b1d-73a0-86e5-577ee285d833-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'recipient': 'wrong@email.com', 'subject': 'Test'}, 'id': 'call_331458', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 34, 

In [38]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': 'Email sent successfully to the corrected recipient with the updated subject and body.', 'extras': {'signature': 'El4KXAFpFH0TqdtuUsXX/z2ICWu0rH5B+WmFQelaVvOTQLninpaFn2WCy5OkYb2P4TJB1go6Vy6G25SUMam/SSM2e6UHvh1CRgepxggxg0FAtAA97k+nxUrARXmAbUAP'}}]


In [39]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='fa5a0ac0-9946-48d5-9a10-2b8b7e5c1bb0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "recipient": "wrong@email.com", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_331458': 'El4KXAFpFH0TJfddnbAKgeYkZV74gTAJiCtHm9MrLsWnUx6awV8eVyeUulNDiJY3LpspWX7WuuKSi0DU/6xvgZ5d7CrDzzqfFP0PRMq7ETDSpn9N3A7jlMAvDV0VofaU'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1d8-4b1d-73a0-86e5-577ee285d833-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'call_331458'}], invalid_tool_calls=[], usage_met